# Trajectory Quantiles Example

This notebook is a researcher-facing exploratory example for inspecting FungMod's trajectory-quantile standard output. It is not an empirical validation, calibration, posterior-uncertainty, or literature-comparison notebook. The example uses existing registry-backed virtual-experiment records, writes standard outputs and reports, then inspects `trajectory_quantiles.csv` guardrails before using the quicklook plot as a presentation layer.

In [ ]:
import os
from pathlib import Path

from fungal_model import environment_grid, virtual_experiment

OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks"))
OUTPUT_DIR = OUTPUT_ROOT / "14_trajectory_quantiles_example"
REGISTRY = Path("data_registry/registry_index.yml")


Create a small public-API virtual experiment from researcher-facing names. This example keeps the environment grid to one metadata-only runtime case so the focus stays on trajectory output inspection rather than environment-response interpretation.

In [ ]:
study = virtual_experiment(
    fungi="beta-glucosidase source",
    substrates="cellobiose substrate",
    environments=environment_grid(temperature_C=[30.0], ph=[5.0], oxygen="aerobic"),
    registry=REGISTRY,
)

[(report.status, report.required_processes) for report in study.preflight(mode="exploratory")]


Run a small exploratory ensemble and write the standard output bundle. `quicklook=False` keeps simulation and table writing separate from presentation; the quicklook figure is generated explicitly from the already written CSV tables below.

In [ ]:
result = study.simulate(
    mode="exploratory",
    n_samples=6,
    seed=18,
    output_dir=OUTPUT_DIR,
    quicklook=False,
)

result.write_summary()
result.write_manifest()
sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))[:8]


Load `trajectory_quantiles.csv` through the public result accessor. These p05/p50/p95 rows are derived from existing `time_series_long.csv` sample rows; they are not validation data, empirical confidence intervals, calibration evidence, or posterior uncertainty.

In [ ]:
trajectory_rows = result.trajectory_quantiles()
trajectory_path = OUTPUT_DIR / "trajectory_quantiles.csv"

assert trajectory_path.exists()
assert trajectory_rows
assert {"p05", "p50", "p95", "trajectory_band_status", "interpretation_guardrail"}.issubset(trajectory_rows[0])
assert {row["source_table"] for row in trajectory_rows} == {"time_series_long"}
assert {row["allowed_use"] for row in trajectory_rows} == {"exploratory_trajectory_summary_not_validation"}
assert all("not validation data" in row["interpretation_guardrail"] for row in trajectory_rows)

[(row["state"], row["time"], row["p05"], row["p50"], row["p95"]) for row in trajectory_rows[:5]]


Generate quicklook figures explicitly from the standard output tables. The trajectory-band image reads `trajectory_quantiles.csv`; it is a browser-friendly inspection aid, not a validation or calibration artifact.

In [ ]:
quicklook_paths = result.write_quicklook_plots(OUTPUT_DIR / "figures")

assert any(Path(path).name == "trajectory_quantile_bands.png" for path in quicklook_paths)
assert (OUTPUT_DIR / "figures" / "trajectory_quantile_bands.png").exists()

[Path(path).name for path in quicklook_paths]


Write the report folder after quicklook generation so the optional HTML index links both the standard trajectory-quantile table and the presentation-only figure.

In [ ]:
report_path = result.write_report(OUTPUT_DIR / "report", include_html=True, include_index=True)
report_dir = OUTPUT_DIR / "report"

assert report_path.exists()
assert (report_dir / "virtual_experiment_report.html").exists()
assert (report_dir / "index.html").exists()

index_text = (report_dir / "index.html").read_text(encoding="utf-8")
assert "trajectory_quantiles.csv" in index_text
assert "trajectory_quantile_bands.png" in index_text

report_path
